# Go deeper · Write it yourself

In the session notebook, every option ran for us. In this notebook, we'll write the code that made it
work: the Query behind the evidence, the transaction that records a decision once, the stream transform
that builds the graph, and the openCypher queries that answer the examiner. Each exercise has a check
that runs your answer and the reference answer on the same data, and a solution you can reveal if you
get stuck.

When an answer passes, it's saved to `solution/`, and the MCP server in the Kiro notebook runs your file
instead of the reference. By the end, the agent that answers the examiner in Kiro is running your code.

The exercises follow the session's four decisions. Not every cell is an exercise: G.10 and G.12 to G.16
run as they are, and so does each "a" cell after a graph exercise, which draws its answer in Neptune's
graph view. You can read and change any of them.

**Cell G.0 · Connect.** Run this cell first. It connects the notebook to your stack, loads the checks
the exercises use, and finds your Carlos evaluation from the session notebook. If it can't find one, it
runs the agent on Carlos once, in under a minute. Every exercise below reads that evaluation.

In [ ]:
# MIT No Attribution
#
# Copyright 2026 Amazon.com, Inc. or its affiliates. All Rights Reserved.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy of this
# software and associated documentation files (the "Software"), to deal in the Software
# without restriction, including without limitation the rights to use, copy, modify,
# merge, publish, distribute, sublicense, and/or sell copies of the Software, and to
# permit persons to whom the Software is furnished to do so.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A
# PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT
# HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
# OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
# SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
#
# SPDX-License-Identifier: MIT-0
# Cell G.0 · connect
from agent.session import connect_deeper, resume
connect_deeper(globals())
CARLOS = resume(globals())

## Decision 1 · The record

Everything the examiner asks about a single decision comes from its partition, so that's where we'll
start.

**Cell G.1 · Read the evidence back.** In the session, Cell 1.3 read Carlos's whole decision with
one Query. Now we'll write that Query ourselves.

**What we write:** the body of `get_evidence(evaluation_id)`. It's a Query on `trace_table` whose
partition key is `EVAL#<evaluation_id>`, and it returns the items.

**Hints**
- Use `KeyConditionExpression=Key("PK").eq(...)`. `Key` is already imported.
- Add `ConsistentRead=True`. Steps written a moment ago must be included, and an eventually consistent
  read could miss them.
- We don't need to sort anything. DynamoDB returns items in sort-key order: `META` first, then
  `STEP#000` onward.

**When it passes,** we see *✅ Your get_evidence returns the same as the reference*, and your
function is saved as `solution/get_evidence.py`. From then on, the MCP server's
`get_decision_evidence` tool runs your code.

In [ ]:
# Cell G.1 · ✏️ the evidence Query
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Read one evaluation's evidence back with one Query on its key.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# The Query returns the steps in sort-key order, so no sorting in Python. The check
# compares your output with the reference's, then saves get_evidence.py.
def get_evidence(evaluation_id):
    return None  # ✏️ one Query on PK = EVAL#<evaluation_id>, strongly consistent

check_get_evidence(get_evidence, CARLOS)

<details>
<summary>🔑 Reveal solution</summary>

```python
def get_evidence(evaluation_id):
    return trace_table.query(
        KeyConditionExpression=Key("PK").eq(f"EVAL#{evaluation_id}"),
        ConsistentRead=True,
    )["Items"]
```

</details>

## Decision 2 · Keys and counts

A count that's updated in the same transaction as the decision is an answer we never have to compute
later. But it only works if a resent decision can't be counted twice, so that's what we'll build
first.

**Cell G.2 · Record a decision once.** In the session, Cell 2.3 showed that separate writes count a
resent decision twice. Now we'll write the transaction that doesn't.

**What we write:** the two actions in the `actions` list. The policy counts are appended for us, and
the cell sends everything as one `transact_write_items` call:
1. A **Put** of the decision item (`TableName`, `Item`), with a condition that no item has its key
   yet: `attribute_not_exists(PK)`. The item has to be in DynamoDB's typed form, so serialize each
   value with `ser.serialize(v)`.
2. An **Update** of the header, which is the item with `SK` = `META`. It sets `status` to `COMPLETE`
   and `completed_at` to `now`, with a condition that `status` is still `IN_PROGRESS`.

**Hint:** `status` is a reserved word in DynamoDB expressions, so refer to it through
`ExpressionAttributeNames`, for example `{"#s": "status"}`, and write `#s` in the expressions.

**When it passes,** the check delivers one decision twice, first through three separate writes and
then through our transaction. Its table shows the separate writes counting the decision twice and
ours counting it once. It also explains why DynamoDB cancelled our second delivery, and ends with
*✅ One decision, recorded once, however often it is delivered*.

In [ ]:
# Cell G.2 · ✏️ one decision, one transaction
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Record a decision and its counts in one transaction.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# Put the decision only if no item has its key; update the header to COMPLETE only if it
# is still IN_PROGRESS ("status" is a reserved word); the policy counts are provided.
def record_decision(evaluation_id, decision, now):
    pk = f"EVAL#{evaluation_id}"
    actions = [
        # ✏️ the write-once Put, and the guarded Update of the header (SK = META)
    ]
    actions += aggregate_updates(TRACE_TABLE_NAME, decision, now)
    dynamodb_client.transact_write_items(TransactItems=actions)

check_decision_transaction(record_decision)

<details>
<summary>🔑 Reveal solution</summary>

```python
def record_decision(evaluation_id, decision, now):
    pk = f"EVAL#{evaluation_id}"
    actions = [
        {"Put": {"TableName": TRACE_TABLE_NAME,
                 "Item": {k: ser.serialize(v) for k, v in decision.items()},
                 "ConditionExpression": "attribute_not_exists(PK)"}},
        {"Update": {"TableName": TRACE_TABLE_NAME,
                    "Key": {"PK": {"S": pk}, "SK": {"S": "META"}},
                    "UpdateExpression": "SET #s = :done, completed_at = :now",
                    "ConditionExpression": "#s = :running",
                    "ExpressionAttributeNames": {"#s": "status"},
                    "ExpressionAttributeValues": {":done": {"S": "COMPLETE"},
                                                  ":running": {"S": "IN_PROGRESS"},
                                                  ":now": {"S": now}}}},
    ]
    actions += aggregate_updates(TRACE_TABLE_NAME, decision, now)
    dynamodb_client.transact_write_items(TransactItems=actions)
```

</details>

**Cell G.3 · The monthly counts.** Question 3 in session Cell 2.2 asked how many decisions failed
HUD 4000.1 II.A.5.d each month. The transaction we just wrote keeps those counts current, so the
answer is a single Query.

**What we write:** the body of `policy_months(policy)`. It's a Query on the partition
`POLICY#<policy>` for sort keys that begin with `MONTH#`, and it returns the whole response, not just
its items.

**Hints:** Combine `Key("PK").eq(...)` and `Key("SK").begins_with("MONTH#")` with `&`. Add
`ReturnConsumedCapacity="TOTAL"` so the check can show us what the Query cost.

**When it passes,** we see *✅ Your policy_months returns the same as the reference*, followed by
about 85 months of counts for under 2 read units. Your function is saved as
`solution/policy_months.py`.

In [ ]:
# Cell G.3 · ✏️ the monthly counts
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Read one policy's monthly counts, and what it cost.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# One Query on PK = POLICY#<policy>, SK beginning with MONTH#, with ReturnConsumedCapacity.
# The check saves policy_months.py.
def policy_months(policy):
    return None  # ✏️

check_policy_months(policy_months)

<details>
<summary>🔑 Reveal solution</summary>

```python
def policy_months(policy):
    return trace_table.query(
        KeyConditionExpression=Key("PK").eq(f"POLICY#{policy}") & Key("SK").begins_with("MONTH#"),
        ReturnConsumedCapacity="TOTAL",
    )
```

</details>

## Decision 3 · The graph

We build the graph from the record, one stream record at a time. That means our transform has to be
safe to run twice, to run out of order, and to run on records it can't parse. In this section, we'll
write that transform and then prove it's safe to replay.

**Cell G.4 · Build the graph from the stream.** This is option B from Decision 3: the Lambda function
that turns each recorded step into nodes and edges on its decision. The function is already written,
except for three `execute_query` calls whose openCypher is blank, and we'll fill in each one.

**openCypher in brief:** a node is written ``(d:Decision {`~id`: $dec_gid})``, with a variable, a
label, and properties, and an edge is written `-[:HAS_DECISION]->` between two nodes. `MERGE` finds the
whole pattern if it exists and creates it if it doesn't, and `ON CREATE SET` sets properties only when
it creates something. `UNWIND` turns a list parameter into one row per item, and `WITH` carries
variables from one part of a query to the next.

**TODO 1, the hub.** First, we `MERGE` the Application, with `~id` `$app_gid`, and the Decision, with
`~id` `$dec_gid`, and then we `MERGE` the `HAS_DECISION` edge between them. We use `ON CREATE SET` for
their plain properties (`id` and `application_id`).

**TODO 2, what a step used.** This applies to every step except the decision. We `MATCH` the Decision
and the Tool (whose `~id` is `"tool:" + $tool`) and `MERGE` a `USED_TOOL` edge. Then we
`UNWIND $policies AS pol`, `MERGE` each Policy (whose `~id` is `"policy:" + pol.id`), and `MERGE` a
`RETRIEVED_POLICY` edge with `issued: pol.issued` in its pattern. We set `version` on the edge
afterward.

**TODO 3, what the decision said.** We `SET` the decision's `outcome`, and then `UNWIND $cited` to
`MERGE` a `CITED_POLICY {reference: c.ref}` edge to each cited Policy. The `VIOLATED` edges need a
query of their own: an `UNWIND` over an empty list ends the query, so a decision with no citations
would never reach them. So we add a second `execute_query` after the first one that does the same over
`$violated`.

**Why we use `MERGE` and record ids everywhere:** streams can deliver a record twice, or out of order.
A `MERGE` on an id taken from the record finds what's already there instead of adding a copy.

**When it passes,** the check has run our transform on three throwaway copies of Carlos's records: the
reference once, ours delivered twice, and ours in reverse order. Its table counts each relationship
type for all three, followed by *✅ Twice and in reverse, your transform builds exactly what the
reference does*. The check deletes the copies afterward.

In [ ]:
# Cell G.4 · ✏️ option B's stream transform
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Three TODOs: the hub MERGE, the retrieval edges, and the decision edges.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# test_transform runs yours on a throwaway copy of Carlos's records, delivered once, twice,
# and in reverse. Ids come from the record, so a replay MERGEs onto what is there.
def project_step(image):
    evaluation_id = image["PK"]["S"].replace("EVAL#", "")
    application_id = image["application_id"]["S"]
    action = image.get("action", {}).get("S", "")
    is_decision = image.get("is_decision", {}).get("BOOL", False)
    dec_gid = f"decision:{evaluation_id}_decision"

    # ✏️ TODO 1: MERGE the Application (`~id` "app:<id>") and the Decision (`~id` dec_gid),
    #    linked by HAS_DECISION, in one query
    execute_query("""
        // ✏️ YOUR MERGE HERE
    """, {"app_gid": f"app:{application_id}", "app_id": application_id, "dec_gid": dec_gid,
          "dec_id": f"{evaluation_id}_decision"})

    if not is_decision:
        # ✏️ TODO 2: MERGE (d)-[:USED_TOOL]->(tool), then UNWIND $policies and MERGE each
        #    (d)-[:RETRIEVED_POLICY {issued}]->(policy), with its version
        execute_query("""
            // ✏️ YOUR MERGE HERE
        """, {"dec_gid": dec_gid, "tool": action, "policies": retrieved_policies(image)})
        return

    # ✏️ TODO 3: the decision's outcome, and its CITED_POLICY and VIOLATED edges (UNWIND each)
    cited = [{"id": normalize_policy_id(r), "ref": r} for r in strings(image, "policies_cited")]
    violated = sorted({normalize_policy_id(r) for r in strings(image, "policies_violated")} - {None})
    execute_query("""
        // ✏️ YOUR SET AND MERGE HERE
    """, {"dec_gid": dec_gid, "outcome": image.get("outcome", {}).get("S", ""),
          "cited": [c for c in cited if c["id"]], "violated": violated})

test_transform(project_step, CARLOS)

<details>
<summary>🔑 Reveal solution</summary>

```python
def project_step(image):
    evaluation_id = image["PK"]["S"].replace("EVAL#", "")
    application_id = image["application_id"]["S"]
    action = image.get("action", {}).get("S", "")
    is_decision = image.get("is_decision", {}).get("BOOL", False)
    dec_gid = f"decision:{evaluation_id}_decision"

    execute_query("""
        MERGE (a:Application {`~id`: $app_gid})
        ON CREATE SET a.id = $app_id, a.application_id = $app_id
        MERGE (d:Decision {`~id`: $dec_gid})
        ON CREATE SET d.id = $dec_id, d.application_id = $app_id
        MERGE (a)-[:HAS_DECISION]->(d)
    """, {"app_gid": f"app:{application_id}", "app_id": application_id, "dec_gid": dec_gid,
          "dec_id": f"{evaluation_id}_decision"})

    if not is_decision:
        execute_query("""
            MATCH (d:Decision {`~id`: $dec_gid})
            MATCH (t:Tool {`~id`: "tool:" + $tool})
            MERGE (d)-[:USED_TOOL]->(t)
            WITH d
            UNWIND $policies AS pol
            MERGE (p:Policy {`~id`: "policy:" + pol.id}) ON CREATE SET p.id = pol.id
            MERGE (d)-[r:RETRIEVED_POLICY {issued: pol.issued}]->(p)
            SET r.version = pol.version
        """, {"dec_gid": dec_gid, "tool": action, "policies": retrieved_policies(image)})
        return

    cited = [{"id": normalize_policy_id(r), "ref": r} for r in strings(image, "policies_cited")]
    violated = sorted({normalize_policy_id(r) for r in strings(image, "policies_violated")} - {None})
    execute_query("""
        MATCH (d:Decision {`~id`: $dec_gid})
        SET d.outcome = $outcome
        WITH d
        UNWIND $cited AS c
        MERGE (p:Policy {`~id`: "policy:" + c.id}) ON CREATE SET p.id = c.id
        MERGE (d)-[:CITED_POLICY {reference: c.ref}]->(p)
    """, {"dec_gid": dec_gid, "outcome": image.get("outcome", {}).get("S", ""),
          "cited": [c for c in cited if c["id"]], "violated": violated})
    execute_query("""
        MATCH (d:Decision {`~id`: $dec_gid})
        UNWIND $violated AS pid
        MERGE (p:Policy {`~id`: "policy:" + pid}) ON CREATE SET p.id = pid
        MERGE (d)-[:VIOLATED]->(p)
    """, {"dec_gid": dec_gid, "violated": violated})
```

</details>

**Cell G.5 · Prove it's safe to replay.** Our transform passed on copies. Next, the check replays
Carlos's real records through the *deployed* transform, and we'll write the query that measures whether
anything changed.

**What we write:** the rest of `FOOTPRINT`, after its first line. It returns every relationship leaving
the decision, counted by type, as `rel` and `n`. We use `OPTIONAL MATCH (d)-[r]->()`, then `type(r)`
and `count(r)`.

**When it passes,** we see the footprint before and after the replay, identical, followed by
*✅ Delivered again, nothing changed*. The check then sends a malformed record in the same batch, and
*✅ A malformed record in the same batch was reported alone* shows that the transform rejects only the
bad record, so one broken item can't hold up the stream.

In [ ]:
# Cell G.5 · the replay proof, then ✏️ its footprint query
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Count everything the transform built for one decision.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# prove_replay_safe delivers the same records to the deployed transform again, then a
# malformed record. Your query is the measuring stick: one row per relationship type
# from the decision ($decision), as rel and n.
FOOTPRINT = """
MATCH (d:Decision {`~id`: $decision})
// ✏️ YOUR PATTERN AND RETURN HERE
"""
prove_replay_safe(FOOTPRINT, CARLOS)

<details>
<summary>🔑 Reveal solution</summary>

```python
FOOTPRINT = """
MATCH (d:Decision {`~id`: $decision})
OPTIONAL MATCH (d)-[r]->()
RETURN type(r) AS rel, count(r) AS n ORDER BY rel
"""
```

</details>

## Decision 4 · Checking what it claims

The runtime checks new decisions as they're made. In this section, we'll write the queries that find
the decisions from before the check existed, and then look more closely at what a citation was about.

**Cell G.6 · Find citations no search returned.** This is session Cell 4.2's query with its key line
removed. The first two `MATCH` lines find the denials that failed II.A.5.d, along with each policy they
cite.

**What we write:** one line, a `WHERE NOT` that keeps a citation only if the same decision has no
`RETRIEVED_POLICY` edge to that policy. In openCypher, a pattern can be a condition, so
`WHERE NOT (d)-[:RETRIEVED_POLICY]->(p)` tests for an edge that isn't there.

**When it passes,** we see *✅ Your query matches the reference*, with about 5,700 rows: the planted
LA-AUDIT-TEST, your annotate copy if you ran session Cell 4.1, and the past denials. The query is saved
as `solution/ungrounded_citations.cypher` for the MCP server's `ungrounded_citations` tool.

In [ ]:
# Cell G.6 · ✏️ WHERE NOT
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Write the line that finds citations no search returned.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# Cell 4.2's query with its WHERE NOT line blank. The check expects LA-AUDIT-TEST and the
# past denials, then saves ungrounded_citations.cypher.
UNGROUNDED = """
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[c:CITED_POLICY]->(p:Policy)
// ✏️ YOUR WHERE NOT HERE
RETURN d.application_id AS application, p.id AS policy, c.reference AS cited_as
"""
check_query("UNGROUNDED", UNGROUNDED, {"ratio_rule": "policy:HUD 4000.1 II.A.5.d"},
                   save_as="ungrounded_citations")

<details>
<summary>🔑 Reveal solution</summary>

```python
UNGROUNDED = """
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[c:CITED_POLICY]->(p:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(p)
RETURN d.application_id AS application, p.id AS policy, c.reference AS cited_as
"""
```

</details>

**Cell G.6a · Your answer, in the graph.** Your `WHERE NOT` found the denials; this cell draws three of
them, the planted one first, with every policy each one retrieved and, separately, the policy it cited
without retrieving. `CALL { ... }` runs both walks from each decision, and `UNION` joins them. Choose
**Graph** in the view.

**What to look for:** a policy reached by `CITED_POLICY` alone, with no `RETRIEVED_POLICY` edge from the
same decision.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Policy":"id"} -l 24 -rel 20
// Cell G.6a · Your WHERE NOT's answer, in the graph: three ungrounded denials
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[:CITED_POLICY]->(cited:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(cited)
WITH DISTINCT d ORDER BY d.application_id = $planted DESC, d.application_id
LIMIT 3
CALL {
  WITH d
  MATCH p = (d)-[:RETRIEVED_POLICY]->(:Policy)
  RETURN p
  UNION
  WITH d
  MATCH p = (d)-[:CITED_POLICY]->(c:Policy)
  WHERE NOT (d)-[:RETRIEVED_POLICY]->(c)
  RETURN p
}
RETURN p

**Cell G.7 · A paragraph-level check.** Carlos's decision cites II.A.5.d down to paragraph (viii). But
did it actually read that paragraph? The query already counts the passages the decision read from
II.A.5.d.

**What we write:** the last column, which counts how many of those passages contain the paragraph's
heading, `$paragraph`. We add up a `CASE WHEN ch.text CONTAINS $paragraph THEN 1 ELSE 0 END` inside
`sum(...)` and name it `with_heading`.

**When it passes,** we see *✅ Your query matches the reference*, with one row. We should read it as a
hint, not proof: the Knowledge Base splits text by size, so a passage doesn't line up exactly with a
paragraph.

In [ ]:
# Cell G.7 · ✏️ a paragraph-level check
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Of the passages Carlos's decision read from II.A.5.d, which hold paragraph viii?
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# For a citation with a paragraph, such as HUD 4000.1 II.A.5.d viii, count the passages the
# decision read from that policy, and how many contain the paragraph's heading: a hint, not
# proof, because passages don't line up with paragraphs.
PARAGRAPH = """
MATCH (d:Decision {`~id`: $decision})-[c:CITED_POLICY]->(p:Policy {`~id`: $policy})
MATCH (d)-[:READ]->(ch:Chunk)
WHERE ch.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS replace(p.id, ' ', '-')
RETURN c.reference AS cited_as, count(ch) AS passages_read,
       // ✏️ YOUR COUNT OF PASSAGES CONTAINING $paragraph HERE
"""
check_query("PARAGRAPH", PARAGRAPH, {"decision": f"decision:{CARLOS}_decision",
            "policy": "policy:HUD 4000.1 II.A.5.d", "paragraph": "viii"})

<details>
<summary>🔑 Reveal solution</summary>

```python
PARAGRAPH = """
MATCH (d:Decision {`~id`: $decision})-[c:CITED_POLICY]->(p:Policy {`~id`: $policy})
MATCH (d)-[:READ]->(ch:Chunk)
WHERE ch.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS replace(p.id, ' ', '-')
RETURN c.reference AS cited_as, count(ch) AS passages_read,
       sum(CASE WHEN ch.text CONTAINS $paragraph THEN 1 ELSE 0 END) AS with_heading
"""
```

</details>

**Cell G.7a · The cited policy and its passages, in the graph.** Your count said how many of the passages
Carlos's decision read come from II.A.5.d. This cell draws them: the decision's `CITED_POLICY` edge to
II.A.5.d, and its `READ` edges to passages whose source document is II.A.5.d (the `WHERE` on each
passage's source URI). Each passage is labeled with the start of its text.

**What to look for:** which passages start with, or contain, the paragraph heading the citation names.

In [ ]:
%%oc -qp params -d {"Decision":"outcome","Policy":"id","Chunk":"text"} -l 24 -rel 20
// Cell G.7a · The policy Carlos's decision cites, and the passages it read from it
MATCH p = (d:Decision {`~id`: $decision})-[:CITED_POLICY]->(:Policy {`~id`: $ratio_rule})
RETURN p
UNION
MATCH p = (d:Decision {`~id`: $decision})-[:READ]->(c:Chunk)
WHERE c.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS 'II.A.5.d'
RETURN p

**Cell G.8 · The nearest passages.** For each passage Carlos's decision read, which other passages are
closest to it in meaning? Neptune Analytics stores each passage's embedding, so we can search from the
passage itself, without calling Bedrock.

**What we write:** the `CALL` and `YIELD` lines, `CALL neptune.algo.vectors.topK.byNode(c, {topK: 4})`
followed by `YIELD node, score`. The next line drops the passage itself, which always comes back as its
own nearest match.

**When it passes,** we see *✅ Your query matches the reference*, with about 40 rows. The scores are
squared Euclidean distances, so lower means closer.

In [ ]:
# Cell G.8 · ✏️ the nearest passages with topK.byNode
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · From each passage Carlos's decision read, find the passages nearest it.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# topK.byNode searches from a node's stored embedding. Distances are squared Euclidean: lower
# is closer, and the passage itself comes back first, so leave it out.
NEAREST = """
MATCH (d:Decision {`~id`: $decision})-[:READ]->(c:Chunk)
// ✏️ YOUR CALL AND YIELD HERE (topK: 4)
WITH c, node, score WHERE id(node) <> id(c)
RETURN id(c) AS passage_read, id(node) AS nearest, score
ORDER BY passage_read, score
"""
check_query("NEAREST", NEAREST, {"decision": f"decision:{CARLOS}_decision"})

<details>
<summary>🔑 Reveal solution</summary>

```python
NEAREST = """
MATCH (d:Decision {`~id`: $decision})-[:READ]->(c:Chunk)
CALL neptune.algo.vectors.topK.byNode(c, {topK: 4})
YIELD node, score
WITH c, node, score WHERE id(node) <> id(c)
RETURN id(c) AS passage_read, id(node) AS nearest, score
ORDER BY passage_read, score
"""
```

</details>

**Cell G.9 · Which denials read an older edition?** HUD issued Update 18 of the handbook on
2026-08-12. Which past denials read II.A.5.d in an earlier edition? Option B puts the edition's issue
date on the `RETRIEVED_POLICY` edge, so the answer is just one hop from the policy.

**What we write:** one `WHERE` line on the edge, which keeps the retrievals whose `r.issued` is before
`$current`.

**When it passes,** we see *✅ Your query matches the reference*, with one row for each earlier
edition. The query is saved as `solution/policy_change_impact.cypher`. The MCP server packages it with
your other answer files, but none of its tools runs it yet.

In [ ]:
# Cell G.9 · ✏️ the edition query
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Past denials that read II.A.5.d before Update 18, by edition.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# Option B keeps the edition on the edge, so it's one hop. Saved as
# policy_change_impact.cypher.
EDITIONS = """
MATCH (p:Policy {`~id`: $policy})<-[r:RETRIEVED_POLICY]-(d:Decision {outcome: 'deny'})
// ✏️ YOUR WHERE ON THE EDGE HERE
RETURN r.issued AS edition_issued, count(DISTINCT d) AS denials
ORDER BY edition_issued
"""
check_query("EDITIONS", EDITIONS, {"policy": "policy:HUD 4000.1 II.A.5.d", "current": "2026-08-12"},
            save_as="policy_change_impact")

<details>
<summary>🔑 Reveal solution</summary>

```python
EDITIONS = """
MATCH (p:Policy {`~id`: $policy})<-[r:RETRIEVED_POLICY]-(d:Decision {outcome: 'deny'})
WHERE r.issued < $current
RETURN r.issued AS edition_issued, count(DISTINCT d) AS denials
ORDER BY edition_issued
"""
```

</details>

**Cell G.9a · The editions, on the edges.** Option B keeps the edition a decision read on its
`RETRIEVED_POLICY` edge, and `-de {"RETRIEVED_POLICY":"issued"}` prints that date on each edge. This cell
takes the three most recent editions before `$current`, and `CALL { ... } LIMIT 2` keeps two past
denials for each.

**What to look for:** one policy node, reached by edges carrying different issue dates. The edition
lives on the relationship, not on the policy, so one query can ask who read which edition.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Policy":"id"} -de {"RETRIEVED_POLICY":"issued"} -l 24 -rel 20
// Cell G.9a · Two past denials for each of the three latest earlier editions, the edition on each edge
MATCH (:Policy {`~id`: $ratio_rule})<-[r:RETRIEVED_POLICY]-(:Decision {outcome: 'deny'})
WHERE r.issued < $current
WITH DISTINCT r.issued AS edition
ORDER BY edition DESC
LIMIT 3
CALL {
  WITH edition
  MATCH p = (d:Decision {outcome: 'deny'})-[r:RETRIEVED_POLICY]->(:Policy {`~id`: $ratio_rule})
  WHERE r.issued = edition
  RETURN p LIMIT 2
}
RETURN p

## Beyond the letter

These last exercises go past the examiner's four asks. First, we'll run the agent on an applicant of
your own, and then we'll write the kind of question that really needs a graph.

**Cell G.10 · Your own applicant.** There's nothing to fill in here. This cell creates an application
from the numbers in it, runs the agent on it in under a minute, and shows the decision. Try changing
the credit score, the debt, or the property value, and run it again to see where the decision turns.
Each run is recorded like every other, so it reaches the graph too.

In [ ]:
# Cell G.10 · your own applicant
# create_application from numbers you choose, then invoke_agent_runtime. Change the credit
# score or the debt, and watch where the decision turns.
from agent import create_application
my_app = create_application(
    applicant_name="Alex Workshop", annual_income=75000, credit_score=660,
    employment_years=3, existing_debt=20000, loan_type="home_purchase",
    amount_requested=300000, term_months=360, property_value=320000,
    property_type="condo", employer="Workshop Corp", position="Engineer")
resp = invoke_agent_runtime(application=my_app)
render_decision(resp, application=my_app)

**Cell G.11 · What needs a graph.** Which other decisions read the same passages as Carlos's? That's a
many-to-many question: each passage leads to the decisions that read it, and each decision read many
passages.

**What we write:** the first line, one `MATCH` with two hops. It goes from Carlos's decision `m` over
`READ` to a passage `c`, and back from `c` over `READ` to another decision, `other`. In openCypher,
that's `(m)-[:READ]->(c)<-[:READ]-(other)`.

**When it passes,** we see *✅ Your query matches the reference*, with the decisions that share the
most passages listed first. In DynamoDB, the same answer would take an item for every passage read, a
GSI on the passage, and join code.

In [ ]:
# Cell G.11 · ✏️ what needs a graph
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Decisions that read the same passages as Carlos's.
#    Change only the lines marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# A many-to-many question: passages to the decisions that read them. In a graph it's two
# hops; in DynamoDB, an item per passage read, a GSI on the passage, and join code.
SAME_PASSAGES = """
// ✏️ YOUR MATCH HERE: (m:Decision {`~id`: $decision}) -READ-> (c:Chunk) <-READ- (other:Decision)
WHERE id(other) <> id(m)
RETURN other.application_id AS application, other.outcome AS outcome,
       count(DISTINCT c) AS shared_passages
ORDER BY shared_passages DESC, application
"""
check_query("SAME_PASSAGES", SAME_PASSAGES, {"decision": f"decision:{CARLOS}_decision"})

<details>
<summary>🔑 Reveal solution</summary>

```python
SAME_PASSAGES = """
MATCH (m:Decision {`~id`: $decision})-[:READ]->(c:Chunk)<-[:READ]-(other:Decision)
WHERE id(other) <> id(m)
RETURN other.application_id AS application, other.outcome AS outcome,
       count(DISTINCT c) AS shared_passages
ORDER BY shared_passages DESC, application
"""
```

</details>

**Cell G.11a · Shared passages, in the graph.** Your query counted the decisions that read the same
passages as Carlos's. This cell draws the three other applicants' decisions that share the most with
it. The first `MATCH` ranks the other decisions by passages in common, leaving out Carlos's own runs and
test copies; the second returns each shared passage as two `READ` edges meeting at it.

**What to look for:** passages with arrows from both sides. Each is a place where two decisions relied
on the same text, so a change to that text reaches all of them.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Chunk":"text"} -l 24 -rel 20
// Cell G.11a · Carlos's decision, and the three other applicants' decisions that read the most of the same passages
MATCH (m:Decision {`~id`: $decision})-[:READ]->(c:Chunk)<-[:READ]-(other:Decision)
WHERE other.application_id <> m.application_id AND NOT other.application_id STARTS WITH 'LA-TEST'
WITH m, other, count(DISTINCT c) AS shared
ORDER BY shared DESC
LIMIT 3
MATCH p = (m)-[:READ]->(:Chunk)<-[:READ]-(other)
RETURN p

## More graph questions

These cells run as they are. We'll read each one, and you can change it and run it again. Each graph
question has an "a" cell after it that draws its answer in Neptune's graph view:

- **G.12:** Carlos's past peers that cite a policy never retrieved, a question nobody keyed, over
  edges that are already there.
- **G.13:** Carlos's steps from DynamoDB, and then his decision's links in the graph.
- **G.14:** the past denials that read II.A.5.d before Update 18.
- **G.15:** the passage that decides Saanvi's case, and which of your runs read it.
- **G.16:** the planted citation's passages, scored against its stated reason by vector similarity:
  on topic isn't the same as read.

**Cell G.12 · Carlos's peers that cite a policy never retrieved.** There's nothing to fill in here.
This cell runs the reference query, which goes from Carlos's three bands to his past peers, and then
applies the same `WHERE NOT` as G.6 to their citations. Nobody keyed this question, but it's still one
statement, because the edges are already in the graph. We see the citations by policy, and how many
peers made each one.

In [ ]:
# Cell G.12 · Carlos's past peers that cite a policy never retrieved
from agent.reference import PEERS_UNGROUNDED
bands = {"credit": "band:credit:580-619", "dti": "band:dti:50-100", "ltv": "band:ltv:96.5-200"}
rows = neptune_query(PEERS_UNGROUNDED, bands)["results"]
display(styled_table(pd.DataFrame(rows), bar_cols=["decisions"],
                     caption="Carlos's past peers: citations no search returned"))
print(f"{sum(r['decisions'] for r in rows)} of Carlos's past peers cite a policy no search returned.")

**Cell G.12a · Peers, through the bands they share.** Three of Carlos's past peers, with their three
`IN_BAND` edges and the policy each one cited without retrieving.

**What to look for:** the three band nodes, with every peer pointing at them. In the graph, peers are
decisions that share the same band nodes, so finding them needs no index of its own.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Policy":"id","Band":"id"} -l 24 -rel 20
// Cell G.12a · Three of Carlos's past peers: the bands they share, and what each cited without retrieving
MATCH (:Band {`~id`: $credit})<-[:IN_BAND]-(d:Decision)-[:IN_BAND]->(:Band {`~id`: $dti})
MATCH (d)-[:IN_BAND]->(:Band {`~id`: $ltv})
MATCH (d)-[:CITED_POLICY]->(c:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(c)
WITH DISTINCT d ORDER BY d.application_id
LIMIT 3
CALL {
  WITH d
  MATCH p = (d)-[:IN_BAND]->(:Band)
  RETURN p
  UNION
  WITH d
  MATCH p = (d)-[:CITED_POLICY]->(c:Policy)
  WHERE NOT (d)-[:RETRIEVED_POLICY]->(c)
  RETURN p
}
RETURN p

**Cell G.13 · Carlos's decision, both ways.** There's nothing to fill in here. First, we see his steps
from one DynamoDB Query, in order. Cell G.13a then shows his decision's links in Neptune's graph view.

In [ ]:
# Cell G.13 · Carlos's steps from DynamoDB, in order (G.13a draws his links in the graph)
show_decision(CARLOS)

**Cell G.13a · Carlos's links, in Neptune's graph view.** It shows his
application, his decision, the tools it called, the policies it failed, and its bands. Choose **Graph**
in the view.

In [ ]:
%%oc -qp params -d {"Application":"id","Decision":"outcome","Policy":"id","Tool":"name","Band":"id"} -l 24 -rel 20
// Cell G.13a · Carlos's decision and what it shares with other decisions
MATCH p = (:Application)-[:HAS_DECISION]->(d:Decision {`~id`: $decision})
RETURN p
UNION
MATCH p = (d:Decision {`~id`: $decision})-[:USED_TOOL|VIOLATED|IN_BAND]->()
RETURN p

**Cell G.14 · The edition question.** There's nothing to fill in here. This is the reference version of
G.9's query: past denials that read an earlier edition of II.A.5.d, grouped by edition. In this
synthetic history, the edition a decision read follows from its date. In production, a Knowledge Base
can lag behind a new edition, or a lender can apply one early, and then only the edge knows which
edition a decision read.

In [ ]:
# Cell G.14 · past denials that read II.A.5.d before Update 18
from agent.reference import EDITIONS
rows = neptune_query(EDITIONS, {"policy": "policy:HUD 4000.1 II.A.5.d", "current": "2026-08-12"})["results"]
display(styled_table(pd.DataFrame(rows), bar_cols=["denials"], caption="Denials by the II.A.5.d edition they read"))
print(f"{sum(r['denials'] for r in rows):,} past denials read an earlier edition of II.A.5.d.")

**Cell G.15 · The passage that decides Saanvi's case.** There's nothing to fill in here. This cell finds
the passage that holds II.A.5.d's 40/50 row (the limits that need two compensating factors), the
passages nearest to it in meaning, and which of your runs on Saanvi's application read each one. It does
a vector search and follows the `READ` edges in one statement. If your runs on Saanvi disagreed, compare
what each one read. Keep in mind that what a run read shows what it relied on, not why it decided.

In [ ]:
# Cell G.15 · the passage that decides Saanvi's case, and who read it
# The passage holding II.A.5.d's "40/50" row, the passages nearest it in meaning, and which of
# your runs on LA-2026-00149 read each one: a vector search and the READ edges in one statement.
READERS = """
MATCH (rule:Chunk) WHERE rule.text CONTAINS '40/50' AND rule.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS 'II.A.5.d'
WITH rule LIMIT 1
CALL neptune.algo.vectors.topK.byNode(rule, {topK: 3})
YIELD node AS passage, score
MATCH (:Application {id: $app})-[:HAS_DECISION]->(d:Decision)
OPTIONAL MATCH (d)-[rd:READ]->(passage)
RETURN id(passage) AS passage, score, d.id AS run, d.outcome AS outcome, count(rd) AS reads
ORDER BY score, run
"""
rows = neptune_query(READERS, {"app": "LA-2026-00149"})["results"]
display(styled_table(pd.DataFrame(rows), caption="Saanvi's runs, by the passages they read"))

**Cell G.15a · Who read the deciding passage.** The vector search finds the three passages nearest
II.A.5.d's 40/50 row; the `MATCH` after it returns each of your Saanvi runs' `READ` edges to them.

**What to look for:** whether every run reached the passages that decide her case, or some didn't.

In [ ]:
%%oc -qp params -d {"Application":"id","Decision":"outcome","Chunk":"text"} -l 24 -rel 20
// Cell G.15a · Saanvi's runs and the passages nearest the 40/50 row that each one read
MATCH (rule:Chunk)
WHERE rule.text CONTAINS '40/50' AND rule.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS 'II.A.5.d'
WITH rule LIMIT 1
CALL neptune.algo.vectors.topK.byNode(rule, {topK: 3})
YIELD node AS passage
MATCH p = (:Application {id: $saanvi})-[:HAS_DECISION]->(:Decision)-[:READ]->(passage)
RETURN p

**Cell G.16 · On topic isn't the same as read.** Nothing to fill in. It reads the planted case from session Cell
4.1; if you skipped that, run `catch_up("plant")` first. Is the planted citation at least *about* the right
thing? This cell scores each cited policy's passages against the planted decision's stated reason,
using cosine similarity, and shows whether the decision actually retrieved that policy. The query uses
Neptune Analytics' built-in vector distance algorithm, which compares each passage's stored embedding
with an embedding of the reason.

In [ ]:
# Cell G.16 · the cited policies' passages, scored against the planted reason
planted = planted_case()
CLOSEST_TEXT = """
MATCH (d:Decision) WHERE id(d) = $decision
MATCH (d)-[:CITED_POLICY]->(p:Policy)
WITH DISTINCT d, p
MATCH (c:Chunk)
WHERE c.`metadata_x-amz-bedrock-kb-source-uri` CONTAINS replace(p.id, ' ', '-')
  AND NOT c.text STARTS WITH '# '
CALL neptune.algo.vectors.distance.byEmbedding(c,
     {embedding: $reason, metric: 'CosineSimilarity'})
YIELD distance
RETURN p.id AS policy, exists((d)-[:RETRIEVED_POLICY]->(p)) AS retrieved,
       distance AS similarity, c.text AS text
ORDER BY similarity DESC
"""
# Neptune takes no parameters inside CALL, so inline() writes the vector into the text.
reason = embed(planted["reason"])
rows = neptune_query(inline(CLOSEST_TEXT, reason=reason),
                     {"decision": planted["decision"]})["results"]
show_closest(rows, planted["reason"])

The cited policy, II.A.5.b, scores close to the planted reason, a short employment history. Its
passages really do ask the lender to verify two years of employment. But no search returned it, so the
decision never had that text in front of it. A similarity score tells us the topic matches, not that
the decision read the passage. Only the record shows what it read.

**Cell G.16a · What it cited, and what it read.** The planted decision's retrieval, the passages it
read, and the policy it cited without retrieving.

**What to look for:** II.A.5.b, connected by `CITED_POLICY` alone, and no passage from it among the ones
the decision read.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Policy":"id","Chunk":"text"} -l 24 -rel 20
// Cell G.16a · What the planted decision retrieved and read, and what it cited without retrieving
MATCH (:Application {id: $planted})-[:HAS_DECISION]->(d:Decision)
CALL {
  WITH d
  MATCH p = (d)-[:RETRIEVED_POLICY|READ]->()
  RETURN p
  UNION
  WITH d
  MATCH p = (d)-[:CITED_POLICY]->(c:Policy)
  WHERE NOT (d)-[:RETRIEVED_POLICY]->(c)
  RETURN p
}
RETURN p